# 03 — Backtest engine

**Workstream C** of the group project plan. This notebook builds the machinery every strategy runs on:
the experiment sampler, the walk-forward loop, the look-ahead guard, the cost model and the failure log.
It produces **no results**. The only strategy it runs is equal weight, as a test fixture. The eight
strategies come in notebook 04 and the 100 runs in notebook 05.

| Task | Output |
| --- | --- |
| C1 | `experiments` frame: 100 (3-year window, sector-stratified asset subset) pairs under the fixed seed |
| C2 | `backtest.run_backtest(prices, weight_fn, cost_model)`, which returns daily returns, weights and turnover |
| C3 | Look-ahead guard: `LookAheadError`, raised inside the loop, plus deliberate-leak tests |
| C4 | Weight drift between rebalances, inside the loop |
| C5 | `backtest.CostModel`, per-asset one-way cost in bp, charged on Σ\|Δw\| |
| C6 | Failure log: a failed optimisation is logged and the run continues on fallback weights |
| C7 | `BacktestResult.turnover`, annualised two-way turnover |
| C8 | Unit tests, all passing |
| C9 | **Table 3.1 — Experiment design**, plus Tables A3.1–A3.2 for the appendix |

Shared code: `src/backtest.py`. Conventions come from
[`docs/project-definitions.md`](../docs/project-definitions.md) §6–7.

## Setup

In [1]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import backtest as bt
from src import config as cfg
from src import data, fmt

np.random.seed(cfg.SEED)  # the sampler uses its own generator, seeded from cfg.SEED

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 120)

---

## Data

The panel is rebuilt exactly as in notebook 01. Nothing is read from disk. The engine needs only the
adjusted-close panel for the universe and the `^IRX` risk-free series, which MSR will use in notebook 04.

**The sampled experiments depend on the panel's last session.** The set of eligible window starts grows
as the panel grows, so the same seed draws different windows on a later extraction date. The 100
experiments are therefore reproducible under the **seed and the extraction date together**. This is one
more reason for plan task G2's single final re-run.

In [2]:
close, volume, extraction_date = data.download_raw()
panel = data.build_panel(close, volume, extraction_date)
prices = panel.prices[cfg.ASSET_ORDER]

print(f"Extraction date : {panel.extraction_date.isoformat()}")
print(f"Panel           : {panel.calendar[0].date()} to {panel.calendar[-1].date()}, "
      f"{len(panel.calendar):,} sessions")
print(f"All-asset start : {panel.full_panel_start.date()}")
print(f"Universe        : {len(cfg.ASSET_ORDER)} assets in {len(cfg.SECTOR_ORDER)} sectors")

Extraction date : 2026-10-07
Panel           : 2011-10-03 to 2026-10-06, 3,774 sessions
All-asset start : 2011-10-03
Universe        : 25 assets in 5 sectors


---

## The walk-forward convention

Each experiment is **756 daily returns**: 504 for estimation (2 years) and 252 for evaluation (1 year).
Only the evaluation year is scored. Trades follow one fixed timing rule:

| Step | Session | What happens |
| --- | --- | --- |
| Formation | close *f* | Weights computed from returns up to and including *f* |
| Execution | close *f* + 1 | Trade at that close; the cost is charged here |
| First return earned | *f* + 2 | The new weights first earn this session's return |

The opening trade executes at the close of the last estimation session, out of cash, so the whole
evaluation year is invested. Three more trades follow at 63-session intervals (quarterly). The first
estimate uses every return up to its formation date, which is the 2-year estimation period less the
one-session lag. Each later estimate uses the trailing 252 returns.

Experiment 1's dates make this concrete:

In [3]:
schedule = bt.Schedule()
experiments = bt.sample_experiments(prices, schedule)

example = experiments.loc[1]
ew = lambda window: np.full(window.shape[1], 1 / window.shape[1])
demo = bt.run_backtest(bt.experiment_prices(prices, example), ew, experiment=1, strategy="EW")

print(f"Experiment 1: {', '.join(example['assets'])}")
print(f"  Returns           {example['start'].date()} to {example['end'].date()}")
print(f"  Estimation ends   {example['estimation_end'].date()}")
print(f"  Evaluation starts {example['evaluation_start'].date()}")
print()
display(demo.trades.assign(
    turnover=lambda f: f["turnover"].map(lambda v: fmt.percent(v * 100, 1)),
    cost=lambda f: f["cost"].map(lambda v: fmt.number(v * 1e4, 2)),
).rename(columns={"formation": "Formation (close f)", "execution": "Execution (close f+1)",
                  "observations": "Returns used", "turnover": "Turnover %", "cost": "Cost bp",
                  "status": "Status"}))

Experiment 1: AAPL, AMZN, PFE, UNH, BAC, GS, KO, PEP, GLD, SLV
  Returns           2016-03-31 to 2019-04-01
  Estimation ends   2018-03-29
  Evaluation starts 2018-04-02



,Formation (close f),Execution (close f+1),Returns used,Turnover %,Cost bp,Status
rebalance,,,,,,
0,2018-03-28,2018-03-29,503,100.0,2.90,ok
1,2018-06-27,2018-06-28,252,7.2,0.21,ok
2,2018-09-26,2018-09-27,252,7.8,0.22,ok
3,2018-12-27,2018-12-28,252,14.6,0.42,ok


The first trade's 100% turnover is the opening trade out of cash. Its cost is charged, but it is
**excluded from reported turnover** because it is the same 100% for every strategy. Turnover then
measures only the rebalancing a strategy chooses to do. The table shows machinery, not a result: equal
weight is a fixture here, and its numbers are reported in notebook 06.

---

## C1 — Experiment sampler

The brief asks for 100 datasets built from random periods **and random subsets of the universe**, so
that the comparison shows whether conclusions depend on which assets are held. Each draw first picks a
subset, then a window.

**The subset: 10 assets, 2 from each sector.** Within each sector the two are drawn uniformly, without
replacement. Three reasons for this design, fixed before any strategy was run:

- **Every portfolio is a diversified multi-asset portfolio.** A uniform draw of 10 from 25 would
  sometimes leave out the real-assets group entirely. The optimisers would then face a very different
  problem from one experiment to the next, and the comparison would partly measure which sectors
  happened to be drawn. Stratifying keeps the sector mix fixed and varies the names.
- **The size keeps estimation honest.** Ten assets on a trailing 252-session window is about 25
  observations per asset. That is enough for the sample covariance to be well conditioned, while still
  giving the optimisers a real problem to solve. The full 25 would mean about 10 per asset, where
  estimation noise dominates.
- **There is still plenty of subset variation.** There are 10⁵ possible subsets, and each asset is
  expected in 40 of the 100 experiments.

**The window** is then drawn uniformly over the sessions where **every asset in that subset** has a
price on every session of the window (definitions §4). In this universe every asset has data for the
whole panel, so the rule never binds, but it stays in the code for any later universe that has short
histories. A (start, subset) pair that has already been drawn is drawn again, so all 100 experiments are
distinct. Subsets are drawn directly, never enumerated, so the sampler scales to any universe size.

The design is set in `src/config.py`: `SUBSET_SIZE` and `SUBSET_GROUPS`. Setting the groups to `None`
gives an unstratified uniform draw instead.

In [4]:
print(f"experiments: {len(experiments)} rows")
display(experiments.assign(
    assets=experiments["assets"].map(", ".join),
    **{c: experiments[c].dt.date for c in ["anchor", "start", "estimation_end", "evaluation_start", "end"]},
).head(10))
print("... first 10 of 100 shown. `anchor` is the session before the first return; its close is the base price.")

experiments: 100 rows


,anchor,start,estimation_end,evaluation_start,end,assets
experiment,,,,,,
1,2016-03-30,2016-03-31,2018-03-29,2018-04-02,2019-04-01,"AAPL, AMZN, PFE, UNH, BAC, GS, KO, PEP, GLD, SLV"
2,2016-04-08,2016-04-11,2018-04-10,2018-04-11,2019-04-10,"AAPL, INTC, JNJ, UNH, WFC, GS, PG, CL, GLD, SLV"
3,2017-10-24,2017-10-25,2019-10-25,2019-10-28,2020-10-26,"AAPL, MSFT, UNH, ABT, WFC, AXP, KO, WMT, VNQ, XOM"
4,2022-11-29,2022-11-30,2024-12-02,2024-12-03,2025-12-04,"MSFT, INTC, UNH, ABT, BAC, WFC, WMT, CL, TLT, XOM"
5,2012-06-19,2012-06-20,2014-06-23,2014-06-24,2015-06-23,"MSFT, ORCL, PFE, ABT, WFC, GS, PEP, WMT, GLD, XOM"
6,2023-04-25,2023-04-26,2025-04-29,2025-04-30,2026-04-30,"AAPL, INTC, JNJ, ABT, BAC, GS, PG, CL, TLT, VNQ"
7,2016-06-15,2016-06-16,2018-06-15,2018-06-18,2019-06-18,"ORCL, AMZN, JNJ, PFE, JPM, WFC, KO, WMT, SLV, XOM"
8,2015-05-05,2015-05-06,2017-05-04,2017-05-05,2018-05-04,"ORCL, AMZN, PFE, UNH, WFC, AXP, PEP, CL, GLD, TLT"
9,2017-01-06,2017-01-09,2019-01-09,2019-01-10,2020-01-09,"MSFT, ORCL, PFE, MRK, WFC, GS, PG, CL, TLT, XOM"


... first 10 of 100 shown. `anchor` is the session before the first return; its close is the base price.


In [5]:
# The sampler's own guarantees, asserted on the drawn frame.
# 1. Reproducible: the same seed on the same panel gives the same 100 pairs.
assert bt.sample_experiments(prices, schedule).equals(experiments)

# 2. A different seed gives a different draw, so the seed is actually used.
assert not bt.sample_experiments(prices, schedule, seed=cfg.SEED + 1).equals(experiments)

# 3. Availability rule: every experiment's block is complete and exactly the right length.
for i, e in experiments.iterrows():
    block = bt.experiment_prices(prices, e)
    assert len(block) == schedule.window + 1, i
    assert block.notna().all().all(), i

# 4. All 100 pairs are distinct, and each subset has the right size, in the fixed asset order.
assert not experiments.duplicated(["start", "assets"]).any()
assert experiments["assets"].map(lambda s: len(s) == cfg.SUBSET_SIZE
                                 and list(s) == sorted(s, key=cfg.ASSET_ORDER.index)).all()

# 5. Stratification: every subset holds exactly its share of every sector.
groups, per_group, n_subsets = bt.subset_rule(cfg.ASSET_ORDER)
if groups is not None:
    for i, subset in experiments["assets"].items():
        counts = {g: sum(t in members for t in subset) for g, members in groups.items()}
        assert set(counts.values()) == {per_group}, (i, counts)

# 6. The design rejects a size that cannot split evenly across the sectors.
try:
    bt.subset_rule(cfg.ASSET_ORDER, subset_size=cfg.SUBSET_SIZE + 1)
    raise AssertionError("an uneven stratified subset size was accepted")
except ValueError:
    pass

print(f"Sampler: reproducible under the seed, complete blocks, 100 distinct pairs, "
      f"{per_group} per sector in every subset ({n_subsets:,} possible subsets).")

Sampler: reproducible under the seed, complete blocks, 100 distinct pairs, 2 per sector in every subset (100,000 possible subsets).


### What the sampler hands the optimisers

The sampler fixes how hard the estimation problem is before any strategy runs, so the size choice
should be checked here, not discovered in notebook 04. For each experiment, the table below takes the
trailing one-year window at the opening trade's formation date: the window every later re-estimate
uses. It reports how well conditioned the correlation matrix is, and the most correlated pair the
subset holds.

In [6]:
rows = []
for i, e in experiments.iterrows():
    r = bt.experiment_prices(prices, e).pct_change().iloc[1:]
    window = r.iloc[schedule.estimation - schedule.lookback: schedule.estimation]
    corr = window.corr()
    eigenvalues = np.linalg.eigvalsh(corr.to_numpy())
    pairs = corr.where(~np.eye(len(corr), dtype=bool)).stack().dropna()
    rows.append({"experiment": i, "condition": eigenvalues[-1] / eigenvalues[0],
                 "max_corr": pairs.max(), "pair": " / ".join(pairs.idxmax())})
conditioning = pd.DataFrame(rows).set_index("experiment")

print(f"Observations per asset in a trailing re-estimate: {schedule.lookback / cfg.SUBSET_SIZE:.0f}")
print(f"Correlation-matrix condition number: median {conditioning['condition'].median():.0f}, "
      f"90th percentile {conditioning['condition'].quantile(0.9):.0f}, max {conditioning['condition'].max():.0f}")
print(f"Most correlated pair in the subset: median {conditioning['max_corr'].median():.2f}, "
      f"max {conditioning['max_corr'].max():.2f}")
print(f"Experiments holding a pair above 0.85: {(conditioning['max_corr'] > 0.85).sum()} of {len(conditioning)}")
print("Pairs above 0.85, by count:", conditioning.loc[conditioning["max_corr"] > 0.85, "pair"].value_counts().to_dict())

Observations per asset in a trailing re-estimate: 25
Correlation-matrix condition number: median 18, 90th percentile 52, max 170
Most correlated pair in the subset: median 0.75, max 0.96
Experiments holding a pair above 0.85: 24 of 100
Pairs above 0.85, by count: {'JPM / BAC': 8, 'JPM / GS': 6, 'BAC / GS': 4, 'JPM / WFC': 2, 'JPM / AXP': 2, 'GS / AXP': 1, 'BAC / WFC': 1}


The 10-asset size passes this check, with one warning carried to notebook 04. A median condition number
around 20 is a well-posed problem. The 25 observations per asset in each trailing estimate are enough
for the sample covariance, though not generous.

**The warning is the banks.** About a quarter of the experiments hold two financials correlated above
0.85, most often `JPM` with `BAC` or `GS`, and the tail of the condition-number distribution comes from
them. Two near-identical assets give an optimiser a weight difference it cannot estimate well. Expect
GMV, MV and MSR to swing between the pair from one rebalance to the next. Notebook 04's D11–D12 checks
and its turnover numbers are where this will show. Stratification does not remove the risk: it fixes
two financials per subset, so a highly correlated pair is always possible.

---

---

## C5 — Transaction-cost model

The cost of a trade is `Σ |Δwᵢ| × bpᵢ`, a fraction of portfolio value charged at the close on which the
trade executes. `Δw` is measured against the **drifted** weights just before the trade, so a strategy
pays only for what it actually changes. The rates are one-way, from definitions §6. Each leg of a trade
(sell one asset, buy another) pays its own asset's rate. The same model at 2× gives the robustness run
in workstream F, and at 0× the gross run.

In [7]:
costs = pd.DataFrame({
    "1× bp": bt.CostModel().rates(cfg.ASSET_ORDER) * 1e4,
    "2× bp": bt.CostModel.stressed().rates(cfg.ASSET_ORDER) * 1e4,
    "0× bp": bt.CostModel.zero().rates(cfg.ASSET_ORDER) * 1e4,
}, index=pd.Index(cfg.ASSET_ORDER, name="Ticker"))
display(fmt.style(costs, {c: lambda v: fmt.number(v, 0) for c in costs}))

# Worked example: move 10% of the portfolio from the cheapest asset to trade into the dearest.
cheap, dear = min(cfg.COSTS_BP, key=cfg.COSTS_BP.get), max(cfg.COSTS_BP, key=cfg.COSTS_BP.get)
trade = pd.Series({cheap: -0.10, dear: 0.10})
print(f"Sell 10% {cheap}, buy 10% {dear}: cost = 0.10 x {cfg.COSTS_BP[cheap]:g}bp + 0.10 x "
      f"{cfg.COSTS_BP[dear]:g}bp = {bt.CostModel()(trade) * 1e4:.1f} bp of portfolio value")

,1× bp,2× bp,0× bp
Ticker,,,
AAPL,3,6,0
MSFT,3,6,0
ORCL,3,6,0
AMZN,3,6,0
INTC,3,6,0
JNJ,3,6,0
PFE,3,6,0
MRK,3,6,0
UNH,3,6,0


Sell 10% GLD, buy 10% AAPL: cost = 0.10 x 2bp + 0.10 x 3bp = 0.5 bp of portfolio value


---

## C3 — The look-ahead guard

`check_no_lookahead` runs inside the loop at every rebalance, on the exact window the weight function is
about to receive. It **raises** `LookAheadError` if any of these fail:

1. the estimation window ends on or before the formation date;
2. the trade executes strictly after the formation date;
3. the new weights first earn a return strictly after the trade.

The failure handler (C6) re-raises `LookAheadError`, so a leak can never be logged as a routine
optimisation failure and the loop continue.

The guard checks what the loop hands over. It cannot see a weight function that reads data through a
closure. That case is covered by test 6 below: the same strategy runs again with every price after a
formation date scrambled, and its weights on that date must not move.

---

## C2, C4, C6, C7 — The loop

`run_backtest(prices, weight_fn, cost_model)` walks one weight function through one experiment.

- **Weights drift (C4).** Each session, `wᵢ ← wᵢ(1 + rᵢ) / (1 + wᵀr)`. They reset only when a trade
  executes.
- **Gross and net from one pass.** Costs never change a weight decision, so the gross path is the
  net path without the cost deductions. A zero-cost run reproduces it exactly (test 2).
- **Failures (C6).** A weight function that raises, or returns weights that break the baseline
  constraints (negative, not summing to 1, non-finite, wrong assets), is logged. The portfolio then
  holds its drifted weights, or equal weight if no previous weights exist. The run is never dropped.
- **Turnover (C7).** Σ|Δw| summed over the three quarterly rebalances and divided by the evaluation
  years. This is two-way turnover per year.
- **Risk-free.** A weight function whose signature names `risk_free` also receives `^IRX` over the
  same window, as a daily decimal rate. Notebook 04's MSR uses this.

Its output, per run:

In [8]:
print("gross, net : daily simple returns,", len(demo.gross), "sessions, evaluation year only")
print("weights    :", demo.weights.shape, "weights at each close, from the opening trade")
print("trades     :", list(demo.trades.columns))
print("failures   :", list(demo.failures.columns))
print(f"turnover   : {demo.turnover * 100:.1f} %/yr (two-way, opening trade excluded)")

gross, net : daily simple returns, 252 sessions, evaluation year only
weights    : (253, 10) weights at each close, from the opening trade
trades     : ['formation', 'execution', 'observations', 'turnover', 'cost', 'status']
failures   : ['experiment', 'date', 'strategy', 'error', 'fallback']
turnover   : 29.6 %/yr (two-way, opening trade excluded)


---

## C8 — Unit tests

The plan asks for three tests. EW on a known panel must reproduce hand-computed wealth. A zero-cost
run must match the no-cost path. Weights must sum to 1 and stay non-negative. Seven more cover the rest
of workstream C, so that each claim in Table 3.1 has a test behind it. The leak tests (5a–5b) are
written to **expect** a raise: they pass only if the guard stops the run.

### The hand-computed ledger (test 1)

Two assets, A and B, with costs of 10 and 20 bp. A shrunken schedule: 2 estimation sessions, 4
evaluation sessions, a rebalance every 2 sessions. Equal weight trades at the close of session 1 (out
of cash) and again at session 3.

| Session | r_A | r_B | Weights held (A / B) | Portfolio return |
| --- | --- | --- | --- | --- |
| 2 | +10% | −5% | 0.5 / 0.5 | 2.50% |
| 3 | +20% | 0% | 0.5366 / 0.4634 (drifted) | 10.73% |
| | | | *rebalance to 0.5 / 0.5 at the close* | |
| 4 | −10% | +10% | 0.5 / 0.5 | 0.00% |
| 5 | +5% | 0% | 0.45 / 0.55 (drifted) | 2.25% |

Buy-and-hold within each quarter gives gross wealth = (0.5 × 1.10 × 1.20 + 0.5 × 0.95 × 1.00) ×
(0.5 × 0.90 × 1.05 + 0.5 × 1.10 × 1.00) = 1.135 × 1.0225 = **1.1605375**.

Costs: the opening trade buys 0.5 of each, so 0.5 × 10 bp + 0.5 × 20 bp = **15 bp**. At the session-3
close the drifted weights are 0.66 / 1.135 = 0.5815 and 0.475 / 1.135 = 0.4185. Selling 0.0815 of A and
buying 0.0815 of B costs 0.0815 × (10 + 20) bp = **2.445 bp**, with a two-way turnover of 16.30%.

In [9]:
def _hand_panel():
    # Prices chosen so the returns are the round numbers in the ledger above.
    dates = pd.bdate_range("2020-01-01", periods=7)
    prices = pd.DataFrame({"A": [100, 100, 100, 110, 132, 118.8, 124.74],
                           "B": [100, 100, 100, 95, 95, 104.5, 104.5]}, index=dates)
    schedule = bt.Schedule(estimation=2, evaluation=4, rebalance_every=2, lookback=2)
    return prices, schedule, bt.CostModel(bp={"A": 10, "B": 20})


def equal_weight(window):
    return np.full(window.shape[1], 1 / window.shape[1])


def inverse_variance(window):
    # A data-dependent fixture: weights move with the estimation window.
    w = 1 / window.var()
    return w / w.sum()


def random_weights(seed):
    rng = np.random.default_rng(seed)
    return lambda window: rng.dirichlet(np.ones(window.shape[1]))


def run_all(weight_fn, cost_model=None, **kwargs):
    return [bt.run_backtest(bt.experiment_prices(prices, e), weight_fn, cost_model,
                            experiment=i, **kwargs) for i, e in experiments.iterrows()]


# ---- 1. EW on a known panel reproduces hand-computed wealth ---------------- #
def test_1_ew_hand_computed():
    p, s, costs = _hand_panel()
    res = bt.run_backtest(p, equal_weight, costs, s)
    np.testing.assert_allclose(res.gross.to_numpy(), [0.025, 0.11 / 1.025, 0.0, 0.0225], atol=1e-12)
    assert np.isclose(res.wealth(net=False).iloc[-1], 1.135 * 1.0225, atol=1e-12)

    drift = 0.66 / 1.135 - 0.5                      # weight A drifted above target
    first, second = 0.5 * 10e-4 + 0.5 * 20e-4, drift * (10e-4 + 20e-4)
    np.testing.assert_allclose(res.trades["cost"], [first, second], atol=1e-12)
    np.testing.assert_allclose(res.trades["turnover"], [1.0, 2 * drift], atol=1e-12)
    net = (1 - first) * 1.135 * (1 - second) * 1.0225
    assert np.isclose(res.wealth().iloc[-1], net, atol=1e-12)
    assert np.isclose(res.turnover, 2 * drift / (4 / cfg.TRADING_DAYS))


# ---- 2. Zero-cost run matches the no-cost path ----------------------------- #
def test_2_zero_cost_matches_gross():
    for res in run_all(inverse_variance, bt.CostModel.zero()):
        assert res.net.equals(res.gross), res.experiment
        assert res.total_cost == 0
    for res in run_all(inverse_variance):
        assert res.total_cost > 0
        assert (res.wealth().to_numpy() <= res.wealth(net=False).to_numpy() + 1e-15).all()


# ---- 3. Weights sum to 1 and stay >= 0 ------------------------------------- #
def test_3_weights_valid():
    for fn in (equal_weight, inverse_variance, random_weights(cfg.SEED)):
        for res in run_all(fn):
            assert np.allclose(res.weights.sum(axis=1), 1, atol=1e-12), res.experiment
            assert (res.weights.to_numpy() >= 0).all(), res.experiment
            assert len(res.gross) == schedule.evaluation


# ---- 4. Weights drift between rebalances and reset only on them (C4) ------- #
def test_4_drift():
    for res, (_, e) in zip(run_all(random_weights(1)), experiments.iterrows()):
        r = bt.experiment_prices(prices, e).pct_change().loc[res.weights.index[1]:]
        w = res.weights
        drifted = w.shift(1).iloc[1:] * (1 + r)
        drifted = drifted.div(drifted.sum(axis=1), axis=0)
        moved = (w.iloc[1:] - drifted).abs().max(axis=1) > 1e-12
        assert set(moved[moved].index) == set(res.trades["execution"].iloc[1:]), res.experiment


# ---- 5. The weight function sees only data up to the formation date (C3) --- #
def test_5_window_timing():
    seen = []
    def spy(window):
        seen.append(window)
        return equal_weight(window)
    res = bt.run_backtest(bt.experiment_prices(prices, example), spy)
    assert [len(w) for w in seen] == [schedule.estimation - 1] + [schedule.lookback] * 3
    for window, (_, t) in zip(seen, res.trades.iterrows()):
        assert window.index[-1] == t["formation"] < t["execution"]
    assert res.gross.index[0] > res.trades["execution"].iloc[0]


# ---- 5a. DELIBERATE LEAK: a window running one session past formation ------ #
def test_5a_leak_raises():
    def leaky(returns, formation, lookback):
        return bt.estimation_window(returns, formation + 1, lookback)   # peeks one day ahead
    try:
        bt.run_backtest(bt.experiment_prices(prices, example), equal_weight, _window_fn=leaky)
    except bt.LookAheadError as exc:
        print(f"    5a raised as designed: {exc}")
        return
    raise AssertionError("a leaky estimation window was not caught")


# ---- 5b. DELIBERATE LEAK: trading on the formation close (lag 0) ----------- #
def test_5b_same_close_execution_raises():
    try:
        bt.run_backtest(bt.experiment_prices(prices, example), equal_weight,
                        schedule=bt.Schedule(lag=0))
    except bt.LookAheadError as exc:
        print(f"    5b raised as designed: {exc}")
        return
    raise AssertionError("same-close execution was not caught")


# ---- 6. Scrambling the future leaves past weights untouched ---------------- #
def test_6_future_invariance():
    rng = np.random.default_rng(0)
    for i in (1, 2, 3):
        block = bt.experiment_prices(prices, experiments.loc[i])
        base = bt.run_backtest(block, inverse_variance)
        for _, t in base.trades.iterrows():
            future = block.index > t["formation"]
            scrambled = block.copy()
            scrambled.loc[future] = block.loc[future] * rng.lognormal(0, 0.05, size=(future.sum(), block.shape[1]))
            again = bt.run_backtest(scrambled, inverse_variance)
            pd.testing.assert_series_equal(again.weights.loc[t["execution"]],
                                           base.weights.loc[t["execution"]])


# ---- 7. Failures are logged and the run is kept (C6) ----------------------- #
def test_7_failures_logged_not_dropped():
    calls = {"n": 0}
    def flaky(window):
        calls["n"] += 1
        if calls["n"] == 1:
            w = np.full(window.shape[1], 1 / window.shape[1])    # breaks long-only, still sums to 1
            w[0], w[1] = w[0] + 0.2, w[1] - 0.2
            return w
        if calls["n"] == 3:
            raise RuntimeError("solver did not converge")
        return inverse_variance(window)
    res = bt.run_backtest(bt.experiment_prices(prices, example), flaky,
                          experiment=1, strategy="TEST")
    assert len(res.gross) == schedule.evaluation and res.net.notna().all()
    assert list(res.failures.columns) == ["experiment", "date", "strategy", "error", "fallback"]
    assert list(res.failures["fallback"]) == [bt.FALLBACK_EW, bt.FALLBACK_HOLD]
    assert list(res.trades["status"]) == ["fallback", "ok", "fallback", "ok"]
    assert res.trades["turnover"].iloc[2] == 0 and res.trades["cost"].iloc[2] == 0   # held, no trade
    assert len(bt.failure_log([res, demo])) == 2


# ---- 8. The cost function matches Σ|Δw| x bp ------------------------------- #
def test_8_cost_function():
    # Four trades across the universe, priced by hand from definitions §6.
    picks = cfg.ASSET_ORDER[::7][:4]
    trades = pd.Series([0.05, -0.20, 0.10, 0.05], index=picks)
    hand = sum(abs(dw) * cfg.COSTS_BP[a] for a, dw in trades.items()) / 1e4
    assert np.isclose(bt.CostModel()(trades), hand)
    assert np.isclose(bt.CostModel.stressed()(trades), 2 * hand)
    try:
        bt.CostModel(bp={picks[0]: 5})(trades)
    except KeyError:
        return
    raise AssertionError("an asset without a cost rate traded for free")


TESTS = [test_1_ew_hand_computed, test_2_zero_cost_matches_gross, test_3_weights_valid, test_4_drift,
         test_5_window_timing, test_5a_leak_raises, test_5b_same_close_execution_raises,
         test_6_future_invariance, test_7_failures_logged_not_dropped, test_8_cost_function]

In [10]:
outcomes = []
for test in TESTS:
    try:
        test()
        outcomes.append((test.__name__, "PASS", ""))
    except Exception as exc:  # report every test, then fail the cell below
        outcomes.append((test.__name__, "FAIL", f"{type(exc).__name__}: {exc}"))

test_report = pd.DataFrame(outcomes, columns=["Test", "Result", "Detail"]).set_index("Test")
display(test_report)
assert (test_report["Result"] == "PASS").all(), "unit tests failed"
print(f"All {len(TESTS)} tests pass.")

    5a raised as designed: Estimation window ends 2018-03-29, after the formation date 2018-03-28.
    5b raised as designed: Trade executes 2018-03-29, not after the formation date 2018-03-29.


,Result,Detail
Test,,
test_1_ew_hand_computed,PASS,
test_2_zero_cost_matches_gross,PASS,
test_3_weights_valid,PASS,
test_4_drift,PASS,
test_5_window_timing,PASS,
test_5a_leak_raises,PASS,
test_5b_same_close_execution_raises,PASS,
test_6_future_invariance,PASS,
test_7_failures_logged_not_dropped,PASS,


All 10 tests pass.


Both deliberate leaks raise `LookAheadError`, as they must. Every other test exercises all 100 sampled
experiments, or the hand panel, rather than a single convenient case.

**Speed.** One strategy across all 100 experiments, and the cost of the exact solver notebook 04 will
use at this subset size:

In [11]:
start = time.perf_counter()
_ = run_all(inverse_variance)
elapsed = time.perf_counter() - start
print(f"100 runs of a closed-form strategy: {elapsed:.2f} s.")

# The optimisers in notebook 04 solve long-only QPs exactly, by enumerating every set of
# held assets: 2^n - 1 small linear systems. Time one at the subset size, on a real window.
from src import strategies as st
window = bt.experiment_prices(prices, example).pct_change().iloc[1:schedule.estimation]
q = window.cov().to_numpy() * cfg.TRADING_DAYS
start = time.perf_counter()
for _ in range(5):
    st.solve_qp(q, np.zeros(len(q)), np.ones(len(q)))
per_solve = (time.perf_counter() - start) / 5
solves = cfg.N_EXPERIMENTS * len(schedule.execution_positions())
print(f"One exact long-only QP at {len(q)} assets ({2 ** len(q) - 1:,} supports): {per_solve * 1e3:.0f} ms, "
      f"so about {per_solve * solves:.0f} s per QP strategy over {solves} rebalances.")

100 runs of a closed-form strategy: 0.19 s.


One exact long-only QP at 10 assets (1,023 supports): 22 ms, so about 9 s per QP strategy over 400 rebalances.


---

## C9 — Table 3.1, experiment design

The fixed parameters first, then what the sampler actually drew. The table goes in report section 4
(*Methods and assumptions*).

In [12]:
design = bt.design_table(experiments, panel.calendar, schedule)

print("Table 3.1 - Experiment design. 100 experiments sampled from the panel "
      f"{panel.calendar[0].date()} to {panel.calendar[-1].date()}, Yahoo Finance, extracted "
      f"{panel.extraction_date.isoformat()}; seed {cfg.SEED}.")
display(design)

Table 3.1 - Experiment design. 100 experiments sampled from the panel 2011-10-03 to 2026-10-06, Yahoo Finance, extracted 2026-10-07; seed 20261001.


,Value,Rationale
Parameter,,
Random seed,20261001,Fixed before any result was seen (definitions §7); numpy default_rng
Experiments,100,The brief's 100 three-year datasets. Every strategy runs on the same 100
Window,756 sessions (3 years),"Contiguous NYSE sessions, chronological order kept"
Estimation / evaluation,504 / 252 sessions,"Brief's suggested split: 2 years in-sample, the 3rd year out-of-sample. Only the 3rd year is scored"
Asset subset,"10 of 25: 2 from each of 5 sectors (100,000 possible subsets)","Stratified by sector, uniform within each, so every portfolio spans every sector and differences between experiments..."
Window start,"Uniform over eligible sessions, given the subset",Eligible means every asset in the subset has a price on every session of the window
First estimate,All returns to the formation date (503 sessions),"Uses the full 2-year estimation period, as the brief proposes"
Re-estimates,"Trailing 252 sessions (1 year), rolling",Brief's proposed update rule: 25 observations per asset at the subset size of 10
Rebalancing,"Every 63 sessions (quarterly), 4 trades per experiment","Opening trade at the close of the last estimation session, then each quarter"


Two appendix tables support the coverage rows. Table A3.1 shows how often each asset is drawn: the
sector-stratified rule gives every asset in a five-asset sector an expected 40 of the 100 experiments. Table A3.2 shows where the evaluation years fall in
calendar time, which workstream F's period split (F2) builds on.

In [13]:
inclusion = bt.inclusion_table(experiments)
print("Table A3.1 - Asset inclusion across the 100 experiments. "
      f"Seed {cfg.SEED}, extracted {panel.extraction_date.isoformat()}.")
display(fmt.style(inclusion, {"Experiments": fmt.count, "Share %": lambda v: fmt.percent(v, 0)}))

years = bt.evaluation_years_table(experiments).to_frame().T
years.index = ["Experiments"]
print("Table A3.2 - Experiments by the calendar year their evaluation period starts. n = 100.")
display(years)

Table A3.1 - Asset inclusion across the 100 experiments. Seed 20261001, extracted 2026-10-07.


,Sector,Experiments,Share %,Earliest start,Median start
Ticker,,,,,
AAPL,Technology,36,36,2011-12-12,2016-04-11
MSFT,Technology,41,41,2011-12-12,2017-01-09
ORCL,Technology,38,38,2011-12-28,2017-01-09
AMZN,Technology,36,36,2011-10-12,2016-10-21
INTC,Technology,49,49,2011-10-12,2016-08-26
JNJ,Healthcare,41,41,2011-10-12,2016-07-26
PFE,Healthcare,39,39,2011-10-12,2016-06-16
MRK,Healthcare,39,39,2011-12-12,2015-07-29
UNH,Healthcare,35,35,2012-02-28,2016-12-19


Table A3.2 - Experiments by the calendar year their evaluation period starts. n = 100.


evaluation_start,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
Experiments,3,8,10,13,9,9,10,7,9,5,6,6,5


**Even coverage, with two stress periods to watch.** Every asset has data for the whole panel, so
the availability rule never binds and window starts are spread across the full eligible range. The
evaluation years run from October 2013 to September 2026, between 5 and 13 per calendar year. Assets
appear in 34–49 experiments each, against an expected 40: sampling noise, not a design bias.

Two stress periods are where workstream F's period split (F2) should look first. **Ten** evaluation
years contain the COVID crash of March 2020, which notebook 02 shows dominates any statistic it enters.
**Fourteen** overlap 2022, the rate shock in which long Treasuries, the asset risk-based strategies favour
most, fell sharply. Results from those experiments can differ from the rest for reasons that have
nothing to do with the strategy.

---

## Done-when checks

The plan's exit criteria for workstream C, asserted rather than eyeballed.

In [14]:
checks = []
passed = dict(zip(test_report.index, test_report["Result"] == "PASS"))

checks.append(("C8 unit tests pass (hand-computed EW, zero-cost = gross, weights valid)",
               all(passed[t] for t in ("test_1_ew_hand_computed", "test_2_zero_cost_matches_gross",
                                      "test_3_weights_valid"))))
checks.append(("C3 deliberate-leak tests raise LookAheadError as designed",
               passed["test_5a_leak_raises"] and passed["test_5b_same_close_execution_raises"]))
checks.append(("C1 experiments frame: 100 rows, required columns, subset design respected",
               len(experiments) == cfg.N_EXPERIMENTS
               and {"start", "end", "assets"} <= set(experiments.columns)
               and experiments["assets"].map(len).eq(cfg.SUBSET_SIZE).all()))
checks.append(("C6 failures are logged and runs kept", passed["test_7_failures_logged_not_dropped"]))
checks.append(("C9 Table 3.1 reports subset design, coverage and start range",
               {"Asset subset", "Experiments per asset", "Window starts", "Evaluation coverage"} <= set(design.index)))

for label, ok in checks:
    print(f"[{'PASS' if ok else 'FAIL'}] {label}")
assert all(ok for _, ok in checks), "workstream C exit criteria not met"

[PASS] C8 unit tests pass (hand-computed EW, zero-cost = gross, weights valid)
[PASS] C3 deliberate-leak tests raise LookAheadError as designed
[PASS] C1 experiments frame: 100 rows, required columns, subset design respected
[PASS] C6 failures are logged and runs kept
[PASS] C9 Table 3.1 reports subset design, coverage and start range


---

## Handoff

Notebook 05 rebuilds the experiments from the seed and runs every strategy through the same loop:

```python
from src import backtest as bt, config as cfg, data

panel = data.build_panel(*data.download_raw())
prices = panel.prices[cfg.ASSET_ORDER]
experiments = bt.sample_experiments(prices)          # the same 100 pairs for every strategy

for i, e in experiments.iterrows():
    block = bt.experiment_prices(prices, e)
    for name, fn in strategies.items():               # from notebook 04
        for label, costs in (("1x", bt.CostModel()), ("2x", bt.CostModel.stressed())):
            res = bt.run_backtest(block, fn, costs, risk_free=panel.risk_free,
                                  experiment=i, strategy=name)
            # res.gross / res.net / res.turnover / res.weights / res.failures
```

| Contract for notebook 04 (D9) | |
| --- | --- |
| Signature | `f(returns_window)`, or `f(returns_window, risk_free)` for MSR |
| Input | Simple daily returns, one column per asset in the subset (`cfg.SUBSET_SIZE`, currently 10); `risk_free` is a daily decimal rate on the same dates |
| Output | A `pd.Series` indexed by those assets, or an array in column order |
| Constraints | Long-only and sum to 1, within 1e-6. Anything else counts as a failure, is logged, and the run continues on fallback weights |
| On failure | Just raise. The loop logs it and falls back, so do not catch errors inside the strategy |